# 05 - Test UI (Drag & Drop Crop Identifier Demo)

Stage 5: Interactive test UI for the trained crop identifier.

Launches a local web page with a drag-and-drop image upload box. Runs the
same leaf-segmentation step from 03_leaf_segmentation before prediction, so
what you see here matches what the model actually receives (and lets you
visually confirm the segmentation is isolating the right leaf before you
trust the prediction).

Install deps:
    pip install gradio torch timm opencv-python albumentations --break-system-packages

Run:
    python 05_test_ui.py
Then open the local URL it prints (usually http://127.0.0.1:7860) --
drag and drop an image straight onto the upload box.

## Imports & Configuration

In [1]:
import json
from pathlib import Path

import cv2
import numpy as np
import torch
import timm
import gradio as gr
import albumentations as A
from albumentations.pytorch import ToTensorV2

MODEL_PATH = Path(r"Z:\Projects\Smart-Farming\models\crop_identifier_v1.pth")
LABELS_PATH = Path(r"Z:\Projects\Smart-Farming\models\crop_identifier_labels.json")
IMG_SIZE = 224
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

eval_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## `load_model_and_labels`

In [2]:
def load_model_and_labels():
    with open(LABELS_PATH) as f:
        classes = json.load(f)
    model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=len(classes))
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model, classes

## Run

In [3]:
MODEL, CLASSES = load_model_and_labels()


# ---- Leaf segmentation (same logic as 03_leaf_segmentation.py) --------
def compute_sharpness_map(gray, ksize=25):
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    lap_sq = lap ** 2
    return cv2.blur(lap_sq, (ksize, ksize))


def green_mask(hsv):
    lower = np.array([25, 30, 30])
    upper = np.array([95, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    return mask


def score_contour(contour, image_shape, sharpness_map):
    h, w = image_shape[:2]
    area = cv2.contourArea(contour)
    if area < 0.005 * h * w:
        return -1, None
    x, y, bw, bh = cv2.boundingRect(contour)
    cx, cy = x + bw / 2, y + bh / 2
    img_cx, img_cy = w / 2, h / 2
    dist_from_center = np.hypot(cx - img_cx, cy - img_cy)
    max_dist = np.hypot(img_cx, img_cy)
    centrality_score = 1 - (dist_from_center / max_dist)
    region_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(region_mask, [contour], -1, 255, thickness=cv2.FILLED)
    mean_sharpness = cv2.mean(sharpness_map, mask=region_mask)[0]
    area_score = area / (h * w)
    combined = (0.45 * area_score) + (0.30 * centrality_score) + (0.25 * min(mean_sharpness / 500, 1.0))
    return combined, (x, y, bw, bh)


def isolate_subject_leaf(image_rgb, padding_ratio=0.08):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    sharpness_map = compute_sharpness_map(gray)
    mask = green_mask(hsv)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    best_score, best_box = -1, None
    for c in contours:
        score, box = score_contour(c, image_rgb.shape, sharpness_map)
        if score > best_score:
            best_score, best_box = score, box
    if best_box is None or best_score < 0.15:
        return None
    x, y, bw, bh = best_box
    pad_x, pad_y = int(bw * padding_ratio), int(bh * padding_ratio)
    h, w = image_rgb.shape[:2]
    x0, y0 = max(0, x - pad_x), max(0, y - pad_y)
    x1, y1 = min(w, x + bw + pad_x), min(h, y + bh + pad_y)
    return image_rgb[y0:y1, x0:x1]


# ---- Prediction ---------------------------------------------------------
def predict(image_rgb):
    """
    image_rgb: numpy array (H, W, 3) RGB, as provided by Gradio's Image
    component. Returns (segmented_leaf_preview, confidence_dict).
    """
    if image_rgb is None:
        return None, {}

    leaf_crop = isolate_subject_leaf(image_rgb)
    used_fallback = leaf_crop is None
    if used_fallback:
        # No confident leaf region found -- fall back to the full image
        # rather than failing outright, but flag it in the label.
        leaf_crop = image_rgb

    tensor = eval_tf(image=leaf_crop)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = MODEL(tensor)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    confidences = {CLASSES[i]: float(probs[i]) for i in range(len(CLASSES))}

    if used_fallback:
        confidences = {"[no leaf detected -- used full image] " + k: v for k, v in confidences.items()}

    return leaf_crop, confidences


demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy", label="Drag & drop a leaf photo here"),
    outputs=[
        gr.Image(type="numpy", label="What the model actually sees (after leaf segmentation)"),
        gr.Label(num_top_classes=5, label="Crop Prediction"),
    ],
    title="Smart Farming - Crop Identifier (Test UI)",
    description=(
        "Upload or drag & drop a leaf/plant photo. The image is first run "
        "through the OpenCV leaf-segmentation step, then classified by the "
        "EfficientNet-B0 crop identifier. If segmentation can't find a "
        "confident leaf region, it falls back to the full image and flags "
        "this in the prediction labels."
    ),
    examples=None,
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
